# 🚀 JAVDL
### Advanced Google Colab Runner

Runs the latest `javdl` branch with secure Colab Secrets, database preflight checks, tests, logging, and Telegram bot startup.

### Required Colab Secrets
- `TELEGRAM_API_ID`
- `TELEGRAM_API_HASH`
- `TELEGRAM_BOT_TOKEN`
- `DATABASE_URL` — Neon PostgreSQL connection string

**Never put Telegram tokens, API hashes, GitHub tokens, or other credentials inside Neon tables.** Secrets stay in Colab Secrets; Neon stores upload-history metadata.

Run cells from top to bottom.

## 🧰 1 · Prepare JAVDL

Clone the latest `javdl` branch into the Colab runtime.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO = "https://github.com/Telebotfaroff/javtiful-scraper.git"
BRANCH = "javdl"
WORKDIR = Path("/content/javdl")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(WORKDIR)],
    check=True,
)
os.chdir(WORKDIR)
print(f"JAVDL {BRANCH} ready: {Path.cwd()}")


## 📦 2 · Install dependencies

Installs FFmpeg, Python packages and Chromium for Playwright fallback extraction.


In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!python -m pip install -q -r requirements.txt
!python -m playwright install --with-deps chromium
!python -m pip install -q pytest


## 🔐 3 · Configure Telegram and Neon

Open **Colab → Secrets** and add:

| Secret | Required |
|---|---|
| `TELEGRAM_API_ID` | Yes |
| `TELEGRAM_API_HASH` | Yes |
| `TELEGRAM_BOT_TOKEN` | Yes |
| `DATABASE_URL` | Yes — Neon connection string |

Credentials are loaded only into the runtime environment. Never store API tokens, hashes, or passwords in database tables.

In [ ]:
from google.colab import userdata

required = ["TELEGRAM_API_ID", "TELEGRAM_API_HASH", "TELEGRAM_BOT_TOKEN", "DATABASE_URL"]
secrets = {}

for name in required:
    try:
        secrets[name] = userdata.get(name)
    except Exception as exc:
        raise RuntimeError(f"Unable to read Colab Secret {name!r}. Add it in Colab Secrets.") from exc

missing = [name for name, value in secrets.items() if not value]
if missing:
    raise RuntimeError("Missing Colab Secrets: " + ", ".join(missing))

if not secrets["TELEGRAM_API_ID"].strip().isdigit():
    raise RuntimeError("TELEGRAM_API_ID must be numeric.")

os.environ.update({name: str(value).strip() for name, value in secrets.items()})
os.environ.update({
    "TELEGRAM_SESSION": "javdl_colab",
    "TELEGRAM_UPLOAD_SESSION": "javdl_uploads",
    "TELEGRAM_UPLOAD_CLIENTS": "2",
    "TELEGRAM_MAX_CONCURRENT_TRANSMISSIONS": "8",
})
print("Secrets loaded into runtime environment (values hidden).")

## 🗄️ 4 · Verify Neon database

This checks the Neon connection and creates the upload-history table/index if needed. It does not print the connection string or store credentials in the database.

In [ ]:
from app.storage.neon_processed import NeonProcessedStore
db = NeonProcessedStore()
db.initialize()
print("✅ Neon connection and upload-history schema are ready.")
print("Upload-history summary:", db.summary())

## 📋 5 · Enable live logs

Useful diagnostics for extraction, downloads, FFmpeg, thumbnails and Telegram uploads.


In [ ]:
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    stream=sys.stdout,
    force=True,
)

for logger_name in ("app", "app.extractor", "app.downloader", "app.uploaders", "app.jobs", "app.bot"):
    logging.getLogger(logger_name).setLevel(logging.INFO)

print("Application logging enabled.")


## 🧪 6 · Run pre-flight check

Compile the application and verify that the Telegram bot factory imports correctly before starting anything.


In [ ]:
!python -m compileall -q app main.py scripts tests
!python -m pytest -q

from app.bot.telegram import create_app

print("JAVDL syntax, imports, and test suite passed.")

## 🤖 7 · Start JAVDL

### Bot flow
`Javtiful URL` → `Preview` → `Download` → `Telegram upload`

Channel/studio URLs can also be sent to the bot for batch downloading.

> **Important:** Colab already has an asyncio event loop. Do not use `app.run()` here.

▶️ Run the cell below and keep it running while you use the bot.


In [ ]:
from pyrogram import idle
from app.bot.telegram import create_app

app = create_app()

print("🚀 Starting JAVDL Telegram bot…")
await app.start()
print("")
print("╔══════════════════════════════════════╗")
print("║        ✅ JAVDL BOT IS ONLINE        ║")
print("╚══════════════════════════════════════╝")
print("")
print("📨 Send a Javtiful URL to the bot.")
print("🛑 Stop with the Colab interrupt button.")

try:
    await idle()
finally:
    if app.is_connected:
        await app.stop()
    print("🛑 JAVDL bot stopped.")
